In [15]:
import pandas as pd
import numpy as np
import shutil
import psrcelmerpy

# input data paths:
mazsim_results_path = 'inputs/mazsim_results/results_2.h5'
mazsim_blocks_path = 'inputs/mazsim_results/blocks_2020_2026-10-07.csv'

orig_soundcast_input_dir = 'R:/e2projects_two/SoundCast/Inputs/dev/landuse/2050/rtp_2026_2050'

# output data paths:
adj_proportions_path = 'inputs/adj_proportions.csv'

In [ ]:
# load mazsim race scenario results
with pd.HDFStore(mazsim_results_path, mode='r') as store:
    h = store['/2050/households']
    p = store['/2050/persons']

mazs = h['maz_id']
p = p.drop(columns=['maz_id'])
p['maz_id'] = p['household_id'].map(mazs)

In [5]:
# Map MAZs to TAZs and add TAZ column to persons dataframe
blk = pd.read_csv(mazsim_blocks_path)
tazs = blk.set_index('maz_id')['zone_id']
p['taz_id'] = p['maz_id'].map(tazs).astype(int)
h['taz_id'] = h['maz_id'].map(tazs).astype(int)

In [6]:
# Calculate the proportion of each race category within each TAZ

race_categories = {
    'white': 'race == 1',
    'black': 'race == 2',
    'native_american': 'race.isin([3,4,5])',
    'asian': 'race == 6',
    'pac_islander': 'race == 7',
    'other': 'race == 8',
    'two_or_more': 'race == 9',
}

race_cat_list = [category for category in race_categories.keys()]

for category, condition in race_categories.items():
    p.eval(f"{category} = {condition}", inplace=True)

p_sum = p.groupby('taz_id')[[category for category in race_categories.keys()]].sum()
taz_sum = p_sum.sum(axis=1)
p_prop = p_sum.div(taz_sum, axis=0).reset_index()

In [7]:
# reshape to long format
prop_out = p_prop.melt(id_vars='taz_id', var_name='control', value_name='proportion')
# add total_col
prop_out['total_col'] = 'pers_taz_weight'
# add group
prop_out['group'] = 'race'
# TAZs with no simulated persons fall back to regional race shares
regional_prop = p_sum.sum() / p_sum.sum().sum()
# add row with regional proportions for TAZs with no simulated persons
regional_row = pd.DataFrame([{'taz_id': None, 'control': category, 'proportion': prop, 'total_col': 'pers_taz_weight', 'group': 'race'} 
                             for category, prop in regional_prop.items()])
prop_out = pd.concat([prop_out, regional_row], ignore_index=True)

In [10]:
# save to csv
prop_out.to_csv(adj_proportions_path, index=False)

In [13]:
# copy hh_and_persons.h5 and parcels_urbansim.txt to input dir
shutil.copy(f'{orig_soundcast_input_dir}/hh_and_persons.h5', 'inputs/hh_and_persons.h5')
shutil.copy(f'{orig_soundcast_input_dir}/parcels_urbansim.txt', 'inputs/parcels_urbansim.txt')

'inputs/parcels_urbansim.txt'

In [ ]:
# download and save TAZ2010 shapefile
eg_conn = psrcelmerpy.ElmerGeoConn()
gdf = eg_conn.read_geolayer('TAZ2010')
gdf.to_file('inputs/taz2010.shp')